# Main

> The FastHTML app: assign numbers to glyphs for the Swift Transcribe → PC path.


In [ ]:
#| default_exp main


In [ ]:
#| export
from fasthtml.common import *
from fastlite import database
import json, unicodedata
from iverson.core import asset_dir, data_dir
from iverson import keys



In [ ]:
#| export
root = asset_dir()
data = data_dir()

app,rt = fast_app(hdrs=(Link(rel='stylesheet', href='/style.css'), Link(rel='icon', href='/icon.png'), Script(src='/app.js')),
                  static_path=str(root/'static'), key_fname=str(data/'.sesskey'))


In [ ]:
#| export
kinds = dict(functions='Functions', monadic_operators='Monadic operators', dyadic_operators='Dyadic operators', syntax_and_literals='Syntax and literals')
table = json.loads((root/'glyph_table.json').read_text())
G,SECS = {},[]
for k,title in kinds.items():
    gs = [dict2obj(dict(sym=r[0], mean=r[1], dev=r[2], iast=r[3], ename=(r[4] if len(r)>4 else ''), sec=title)) for r in table[k]]
    gs = [g for g in gs if G.setdefault(g.sym, g) is g]
    if gs: SECS.append((title, gs))

SP = {'space':'␣ Space', 'return':'⏎ Return', 'tab':'⇥ Tab', 'backspace':'⌫ Backspace', 'escape':'⎋ Escape',
      'left':'← Left', 'right':'→ Right', 'up':'↑ Up', 'down':'↓ Down'}
bad = {'Cc','Cf','Cn','Co','Cs'}
def typable(t): return 0 < len(t) <= 8 and not any(c.isspace() or unicodedata.category(c) in bad for c in t)
def typed(sym): return sym.split(',')[0].replace('…','').strip() or sym
def lab(k): return SP.get(k, k)
def cps(k): return '' if k in SP else ' '.join(f'U+{ord(c):04X}' for c in k)



In [ ]:
#| export
class Sound: id:int=None; name:str=''; glyph:str=''; key:str=''; samples:bytes=b''
class Setting: key:str=''; value:str=''
db = database(str(data/'gnn.db'))
sounds = db.create(Sound, pk='id')
settings = db.create(Setting, pk='key')

def snd(sym): return first(s for s in sounds() if s.glyph==sym)
def skey(s): return s.key if s.key in SP or typable(s.key) else (typed(s.glyph) if s.glyph in G else '')

def talk_key():
    "Hold-to-talk key for `iverson-receive` (default ctrl)."
    from iverson.keys import normalize_talk_key
    row = first(x for x in settings() if x.key == 'talk_key')
    return normalize_talk_key(getattr(row, 'value', None) if row else 'ctrl')

def set_talk_key(v):
    "Persist talk key; returns canonical id."
    from iverson.keys import normalize_talk_key, TALK_KEY_LABELS
    v = normalize_talk_key(v)
    allowed = {k for k,_ in TALK_KEY_LABELS}
    if v not in allowed: v = 'ctrl'
    existing = first(x for x in settings() if x.key == 'talk_key')
    if existing: settings.update(key='talk_key', value=v)
    else: settings.insert(Setting(key='talk_key', value=v))
    return v

CODE_LEN = 3  # Sound Names codes are always 3 digits (001–999)

def taken_names(except_id=None):
    return {s.name for s in sounds() if except_id is None or s.id != except_id}

def normalize_code(name):
    "Pad/trim to a 3-digit Sound Names code, or None if invalid."
    name = (name or '').strip()
    if not name.isdigit(): return None
    if len(name) > CODE_LEN: return None
    return name.zfill(CODE_LEN)

def next_code():
    "Next free 3-digit code (001, 002, …)."
    used = taken_names()
    for i in range(1, 10 ** CODE_LEN):
        c = str(i).zfill(CODE_LEN)
        if c not in used: return c
    return '001'

def set_number(sym, name):
    "Assign a 3-digit number name to a glyph. Creates the sound row if needed. Returns (msg, err)."
    name = normalize_code(name)
    if name is None: return f'Enter a number up to {CODE_LEN} digits (stored as {CODE_LEN}-digit code, e.g. 001).', True
    other = first(s for s in sounds() if s.name == name and s.glyph != sym)
    if other: return f'Number {name} is already used by {other.glyph or other.name}.', True
    if s := snd(sym):
        sounds.update(id=s.id, name=name)
        return f'{sym} is number {name}. Speak those {CODE_LEN} digits in iverson.io to type it.', False
    sounds.insert(Sound(name=name, glyph=sym, samples=b''))
    return f'{sym} assigned number {name}. Speak those {CODE_LEN} digits in iverson.io to type it.', False

def hue(i, n): return f'--t:{i/max(n-1,1):.3f}'


In [ ]:
#| export
HERO = dict(keys=('Speak in notations.', 'Say a number in iverson.io, and its glyph is typed.'),
            names=('Number each glyph.', 'Enter a number for a glyph; speak that number in the Swift app to type it.'))

def page(cur, *c):
    tabs = [('keys','/','Sound Keyboard'), ('names','/names','Sound Names')]
    return Title('iverson.io'), Div(
        Nav(Ul(Li(Img(src='/icon.png', height=28), Strong(' iverson.io'))),
            Ul(*[Li(A(t, href=h, aria_current='page' if k==cur else None)) for k,h,t in tabs])),
        Header(H1(HERO[cur][0]), P(HERO[cur][1]), cls='hero'), Main(*c), cls='container')

def Notice(msg='', err=False): return Div(msg, id='notice', cls='err' if err else 'ok', hx_swap_oob='true')

def KeyPick(s, ctx):
    gk = typed(s.glyph) if s.glyph in G else ''
    v,tgt = dict(id=s.id, ctx=ctx), dict(det='#det', row=f'#r{s.id}', kbd=f'#k{s.id}')[ctx]
    kw = dict(name='v', hx_post='/setkey', hx_trigger='change', hx_target=tgt, hx_swap='outerHTML')
    return Span(Input(value='' if s.key in SP else s.key, placeholder=gk or 'Key', maxlength=8, hx_vals=dict(v, f='t'), **kw),
                Select(Option(f'Glyph ({gk})' if gk else 'No key', value='-'), *[Option(l, value=n, selected=s.key==n) for n,l in SP.items()], hx_vals=dict(v, f='s'), **kw), cls='kp')

@rt
def setkey(id:int, f:str, v:str='', ctx:str='row'):
    s,v = sounds[id], v.strip()
    if f=='s': k = '' if v=='-' else v
    elif not v: k = s.key if s.key in SP else ''
    else: k = v if typable(v) and v not in SP else s.key
    sounds.update(id=id, key=k)
    if ctx=='det': return Detail(s.glyph)
    if ctx=='kbd': return KRow(sounds[id], *ix(sounds[id]))
    return Row(sounds[id], *ix(sounds[id]))

def ix(s): ids = [x.id for x in sounds()]; return ids.index(s.id), len(ids)



In [ ]:
#| export
def NameForm(sel='', oob=False):
    s = snd(sel) if sel else None
    opts = [Option('Choose a glyph…', value='', selected=not sel)]
    for title, gs in SECS:
        opts.append(Optgroup(*[Option(f'{g.sym}  {g.ename} — {g.mean}', value=g.sym, selected=g.sym==sel) for g in gs], label=title))
    return Form(
        Label('Glyph', Select(*opts, name='sym', required=True)),
        Label('Number', Input(name='name', value=s.name if s else next_code(), placeholder='e.g. 001', inputmode='numeric',
                              pattern='[0-9]{1,3}', maxlength='3', required=True, autocomplete='off')),
        Button('Assign', type='submit', cls='lg'),
        hx_post='/assign', hx_target='#lib', hx_swap='outerHTML',
        cls='bar num-form', id='nameform', **(dict(hx_swap_oob='true') if oob else {}))

def Detail(sym=''):
    g,s = G.get(sym), snd(sym)
    if not g: return Div(id='det')
    info = (Small(f'number {s.name} → types {lab(skey(s))}'), KeyPick(s, 'det'),
            Button('Forget', cls='outline', hx_post='/forget', hx_vals=dict(id=s.id, view='g', sel=sym), hx_target='#lib', hx_swap='outerHTML')
           ) if s else (Small('No number yet — use the form above.'),)
    return Div(Div(g.sym, cls='big'), Div(Strong(g.ename or g.mean), Br(), g.mean, Br(), Small(g.sec)), Div(*info, cls='acts'), id='det')

def Tile(g, i, n, sel, s):
    cls = 'taught' if s else ''
    return Div(Small(f'#{s.name}' if s else g.mean), B(g.sym), I(g.ename or g.mean), cls=f'tile {cls} {"sel" if g.sym==sel else ""}', style=hue(i,n), role='button', tabindex=0,
               data_sym=g.sym, title=f'{g.sym}  {g.ename} — {g.mean}' + (f'  #{s.name}' if s else ''))

def Row(s, i, n):
    return Div(Span('~'), Span(s.glyph),
               Input(value=s.name, name='v', placeholder='Number', inputmode='numeric',
                     hx_post='/rename', hx_vals=dict(id=s.id), hx_trigger='change', hx_swap='none'),
               Small(f'types {lab(skey(s))}'), KeyPick(s, 'row'),
               Button('×', cls='outline', title='Forget this mapping', hx_post='/forget', hx_vals=dict(id=s.id, view='l'), hx_target='#lib', hx_swap='outerHTML'),
               id=f'r{s.id}', cls='row', style=hue(i,n))

def Lib(view='g', sel=''):
    tabs = Div(*[Button(t, cls='' if view==v else 'outline', hx_get='/lib', hx_vals=dict(view=v, sel=sel), hx_target='#lib', hx_swap='outerHTML') for v,t in (('g','Glyphs'), ('l','All numbers'))], role='group', cls='tabs')
    if view == 'g':
        by = {s.glyph:s for s in sounds() if s.glyph}
        body = (*[Div(H4(t, Small(f'  {sum(g.sym in by for g in gs)} of {len(gs)} numbered')),
                      Div(*[Tile(g, i, len(gs), sel, by.get(g.sym)) for i,g in enumerate(gs)], cls='tiles')) for t,gs in SECS], Detail(sel))
    else:
        ss = sounds()
        body = (H4(f'Numbered glyphs ({len(ss)})', Button('Remove All', cls='outline', hx_post='/forgetall', hx_target='#lib', hx_swap='outerHTML', hx_confirm='Forget every mapping?')),
                *[Row(s, i, len(ss)) for i,s in enumerate(ss)]) if ss else (P('No numbers yet. Pick a glyph and enter a number for it.'),)
    return Div(tabs, *body, id='lib', data_view=view, data_sel=sel)





In [ ]:
#| export
@rt
def lib(view:str='g', sel:str=''): return Lib(view, sel)
@rt
def detail(sym:str): return Detail(sym)

@rt('/names')
def names():
    return page('names',
        NameForm(),
        P('Speak the number in iverson.io (Link on). ', Code('iverson-receive'), ' types the glyph.'),
        Div(id='notice'), Lib())

@rt
def assign(sym:str, name:str='', view:str='g', sel:str=''):
    msg, err = set_number(sym, name)
    return Lib(view, sel or sym), Notice(msg, err), NameForm(sel or sym, oob=True)

@rt
def rename(id:int, v:str):
    v = normalize_code(v)
    if v is None: return
    other = first(s for s in sounds() if s.name == v and s.id != id)
    if other: return
    sounds.update(id=id, name=v)

@rt
def forget(id:int, view:str='g', sel:str=''): sounds.delete(id); return Lib(view, sel), Notice('', False)
@rt
def forgetall(): [sounds.delete(s.id) for s in sounds()]; return Lib('l', ''), Notice('', False)

def KRow(s, i, n):
    k = skey(s)
    return Div(Span(s.glyph), Strong(s.name), Small(cps(k)), Small('number'), KeyPick(s, 'kbd'), id=f'k{s.id}', cls='row', style=hue(i,n)+('' if k else ';opacity:.55'))

def TalkKeyPanel():
    from iverson.keys import TALK_KEY_LABELS
    cur = talk_key()
    return Div(
        Strong('Talk key'),
        P('Hold this key on the Mac while you speak in iverson.io; release to type into the focused field.'),
        Form(
            Label('Key', Select(*[Option(lab, value=k, selected=k==cur) for k,lab in TALK_KEY_LABELS], name='v',
                               hx_post='/talkkey', hx_trigger='change', hx_target='#talkkey', hx_swap='outerHTML')),
            Small(f'Current: {dict(TALK_KEY_LABELS).get(cur, cur)}'),
            id='talkkey-form'),
        id='talkkey', cls='panel')

@rt
def talkkey(v:str='ctrl'):
    set_talk_key(v)
    return TalkKeyPanel()

@rt('/')
def index():
    ss = sounds()
    return page('keys',
        Div(Div(Strong('Sound keyboard'), Small('Speak a number in iverson.io (Link on). '), Code('pixi run receive --code …'), Small(' types the glyph.')), cls='panel'),
        TalkKeyPanel(),
        Div(Div(H4('Numbers and what they type'), *[KRow(s, i, len(ss)) for i,s in enumerate(ss)]) if ss else
            P('No numbers yet. Assign a number to each glyph on the Sound Names page.'), cls='stack'))



In [ ]:
#| export

#| export
@rt('/sys/status')
def sysstatus():
    "Accessibility (typing) status for receive."
    return keys.status()

@rt('/sys/prompt')
def sysprompt():
    "Ask macOS for Accessibility so receive can type into other apps."
    keys.prompt(); return keys.status()

@rt('/sys/bluetooth')
def sysbluetooth():
    "Open Bluetooth settings (receive needs BT on to find iverson.io)."
    keys.open_bluetooth(); return dict(ok=True)

from iverson.api import mount
mount(app, rt, globals())

def main():
    "Run the web app on http://127.0.0.1:5001"
    import uvicorn
    uvicorn.run(app, host='127.0.0.1', port=5001)



In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()
